In [8]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import time

# Paths
PROJECT_ROOT = Path("..").resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MAPPINGS_DIR = PROJECT_ROOT / "data" / "mappings"

# Load database credentials
load_dotenv(PROJECT_ROOT / ".env")

DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "nfhs_health_analytics")
DB_USER = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "")

conn_string = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(conn_string)

with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database()"))
    print(f"Connected to: {result.fetchone()[0]}")
    
print("Setup complete.")

Connected to: nfhs_health_analytics
Setup complete.


In [9]:
# Loading all csv files

df_wide = pd.read_csv(PROCESSED_DIR / "nfhs5_districts_wide.csv")
df_long = pd.read_csv(PROCESSED_DIR / "nfhs_districts_long.csv")
df_district_meta = pd.read_csv(MAPPINGS_DIR / "district_metadata.csv")
df_indicator_meta = pd.read_csv(MAPPINGS_DIR / "indicator_metadata.csv")

print("Files loaded:")
print(f"  nfhs5_districts_wide:  {df_wide.shape}")
print(f"  nfhs_districts_long:   {df_long.shape}")
print(f"  district_metadata:     {df_district_meta.shape}")
print(f"  indicator_metadata:    {df_indicator_meta.shape}")

Files loaded:
  nfhs5_districts_wide:  (647, 110)
  nfhs_districts_long:   (66976, 12)
  district_metadata:     (647, 6)
  indicator_metadata:    (105, 5)


In [10]:
print("Loading dim_districts...")
start = time.time()

df_district_meta.to_sql(
    name='dim_districts',
    schema='nfhs',
    con=engine,
    if_exists='append',
    index=False,
    method='multi',          # Faster bulk insert
    chunksize=500
)

elapsed = time.time() - start
print(f"dim_districts loaded: {len(df_district_meta)} rows ({elapsed:.1f}s)")


with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM nfhs.dim_districts"))
    print(f"   Verification — rows in DB: {result.fetchone()[0]}")

Loading dim_districts...
dim_districts loaded: 647 rows (0.4s)
   Verification — rows in DB: 647


In [11]:
print("Loading dim_indicators...")
start = time.time()

df_indicator_meta.to_sql(
    name='dim_indicators',
    schema='nfhs',
    con=engine,
    if_exists='append',
    index=False,
    method='multi',
    chunksize=500
)

elapsed = time.time() - start
print(f"dim_indicators loaded: {len(df_indicator_meta)} rows ({elapsed:.1f}s)")

# Verify
with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM nfhs.dim_indicators"))
    print(f"   Verification — rows in DB: {result.fetchone()[0]}")

Loading dim_indicators...
dim_indicators loaded: 105 rows (0.1s)
   Verification — rows in DB: 105


In [12]:
print("Loading fact_health_long...")
print(f"  Total rows to insert: {len(df_long):,}")
start = time.time()

df_long.to_sql(
    name='fact_health_long',
    schema='nfhs',
    con=engine,
    if_exists='append',
    index=False,
    method='multi',
    chunksize=1000
)

elapsed = time.time() - start
print(f"fact_health_long loaded: {len(df_long):,} rows ({elapsed:.1f}s)")

# Verify
with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM nfhs.fact_health_long"))
    print(f"   Verification — rows in DB: {result.fetchone()[0]:,}")

Loading fact_health_long...
  Total rows to insert: 66,976
fact_health_long loaded: 66,976 rows (54.9s)
   Verification — rows in DB: 66,976


In [13]:
print("Loading district_health_wide (wide format)...")
start = time.time()

with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS nfhs.district_health_wide CASCADE"))
    conn.commit()

# Let pandas create the table with correct column types
df_wide.to_sql(
    name='district_health_wide',
    schema='nfhs',
    con=engine,
    if_exists='replace',
    index=False,
    method='multi',
    chunksize=500
)

elapsed = time.time() - start
print(f"district_health_wide loaded: {df_wide.shape[0]} rows × {df_wide.shape[1]} cols ({elapsed:.1f}s)")

# Verify
with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM nfhs.district_health_wide"))
    count = result.fetchone()[0]
    
    result2 = conn.execute(text("""
        SELECT column_name 
        FROM information_schema.columns 
        WHERE table_schema = 'nfhs' AND table_name = 'district_health_wide'
        ORDER BY ordinal_position
    """))
    columns = [row[0] for row in result2.fetchall()]
    
    print(f"   Verification — rows: {count}, columns: {len(columns)}")
    print(f"   First 10 columns: {columns[:10]}")

Loading district_health_wide (wide format)...
district_health_wide loaded: 647 rows × 110 cols (4.6s)
   Verification — rows: 647, columns: 110
   First 10 columns: ['state', 'state_census_code', 'district_name', 'district_id', 'district_census_code', 'region', 'anaemia_all_women', 'anaemia_children', 'anaemia_nonpregnant_women', 'anaemia_pregnant_women']


In [15]:
print("Creating indexes on wide table...")

index_queries = [
    "CREATE INDEX IF NOT EXISTS idx_wide_state ON nfhs.district_health_wide(state)",
    "CREATE INDEX IF NOT EXISTS idx_wide_district ON nfhs.district_health_wide(district_name)",
    "CREATE INDEX IF NOT EXISTS idx_wide_region ON nfhs.district_health_wide(region)",
]

with engine.connect() as conn:
    for query in index_queries:
        conn.execute(text(query))
    conn.commit()

print("Indexes created on district_health_wide")

Creating indexes on wide table...
Indexes created on district_health_wide


In [16]:
print("=" * 60)
print("DATABASE LOAD VERIFICATION")
print("=" * 60)

tables_to_check = [
    ('nfhs.dim_districts', len(df_district_meta)),
    ('nfhs.dim_indicators', len(df_indicator_meta)),
    ('nfhs.fact_health_long', len(df_long)),
    ('nfhs.district_health_wide', len(df_wide)),
]

all_good = True
with engine.connect() as conn:
    for table_name, expected_rows in tables_to_check:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table_name}"))
        actual_rows = result.fetchone()[0]
        match = "Yes" if actual_rows == expected_rows else "No"
        if actual_rows != expected_rows:
            all_good = False
        print(f"  {match} {table_name}: {actual_rows:,} rows (expected {expected_rows:,})")

print()
if all_good:
    print("All tables loaded successfully!")
else:
    print("Row count mismatch")

DATABASE LOAD VERIFICATION
  Yes nfhs.dim_districts: 647 rows (expected 647)
  Yes nfhs.dim_indicators: 105 rows (expected 105)
  Yes nfhs.fact_health_long: 66,976 rows (expected 66,976)
  Yes nfhs.district_health_wide: 647 rows (expected 647)

All tables loaded successfully!


In [18]:
print("TESTING QUERIES FROM PYTHON")

# Test View 1: State summary
df_state = pd.read_sql("SELECT * FROM nfhs.vw_state_summary", engine)
print(f"\nState Summary (top 5 by stunting):")
print(df_state[['state', 'num_districts', 'avg_stunting', 'avg_anaemia_children', 
                'avg_full_vaccination']].head().to_string(index=False))

# Test View 2: CRITICAL districts
df_critical = pd.read_sql(
    "SELECT * FROM nfhs.vw_child_nutrition WHERE nutrition_risk_category = 'CRITICAL'", 
    engine
)
print(f"\nDistricts in CRITICAL nutrition risk: {len(df_critical)}")

# Test View 3: Regional comparison
df_region = pd.read_sql("""
    SELECT region, COUNT(*) as districts,
           ROUND(AVG(child_stunting)::NUMERIC, 1) as avg_stunting,
           ROUND(AVG(anaemia_children)::NUMERIC, 1) as avg_anaemia
    FROM nfhs.district_health_wide
    GROUP BY region ORDER BY avg_stunting DESC
""", engine)
print(f"\nRegional Comparison:")
print(df_region.to_string(index=False))

TESTING QUERIES FROM PYTHON

State Summary (top 5 by stunting):
               state  num_districts  avg_stunting  avg_anaemia_children  avg_full_vaccination
           Meghalaya             11          44.2                  41.8                  69.6
               Bihar             38          42.6                  70.1                  72.4
Dadra & Nagar Haveli              1          42.4                  74.8                 100.0
           Jharkhand             24          40.2                  68.2                  75.2
       Uttar Pradesh             71          39.8                  66.5                  69.4

Districts in CRITICAL nutrition risk: 127

Regional Comparison:
   region  districts  avg_stunting  avg_anaemia
     East        112          37.5         68.2
     West         66          36.2         73.8
  Central         68          35.5         71.2
    North        203          32.5         68.0
Northeast         90          32.1         55.8
    South        10